In [2]:
from pathlib import Path 
import os 
import sys
sys.path.insert(0, "../../../../../shared_utils")
from experiment_utils import manual_filtering

import matplotlib.pyplot as plt 
import scanpy as sc
import numpy as np 
import pandas as pd
import scipy as sp 
import seaborn as sns
from tqdm import tqdm 
import copy
import math
import yaml
import fpsample

## Read annotation file

In [3]:
# Annotation file for protocols 
ANNOTATION_CFG_FILE = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/annotation/default.yaml"
with open(ANNOTATION_CFG_FILE, "r") as fb:
    annotation_cfg = yaml.safe_load(fb)
protocol_cols = annotation_cfg["protocol_columns"]

ANNOTATION_CFG_FILE_BOUNDS = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/constraints/default_all_cols.yaml"
with open(ANNOTATION_CFG_FILE_BOUNDS, "r") as fb:
    annotation_cfg_bound = yaml.safe_load(fb)
ubound_dict, lbound_dict = annotation_cfg_bound["ubound_dict"],  annotation_cfg_bound["lbound_dict"]

# Collect per parameter bounds 
bounds = {}
for param in lbound_dict:
    bounds[param] = (lbound_dict[param], ubound_dict[param])

### Collect optimized samples 

In [4]:
path_to_results = Path("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/bloodplus_inverse/")
paths = os.listdir(path_to_results)

In [12]:
cell_types = ["CD14Mono",
                "CD16Mono",
                "CyclingPro1",
                "CyclingPro2",
                "EosBasMast1",
                "EosBasMast2",
                "EosBasMast3",
                "EryPro",
                "HSCs",
                "ProB",
                "late_MgkPro",
                "GMP-Neutro",
                "preProB",
                "pre_cDC1"]

In [7]:
pure_cell_type_solution_dict = {cell_type: [] 
                                for cell_type in cell_types
                               }

for experiment_folder in tqdm(paths):
    # Only consider pure populations 
    if "pure" in experiment_folder:
        cell_type_file_paths = path_to_results / experiment_folder
        # For all cell types 
        for cell_type in os.listdir(cell_type_file_paths):
            config_exp_folders = cell_type_file_paths / cell_type
            # For all the experiments in the folders 
            for config_exp_folder in os.listdir(config_exp_folders):
                # Be sure candidates are there 
                if "candidates.csv" in os.listdir(config_exp_folders / config_exp_folder):
                    path = config_exp_folders / config_exp_folder / "candidates.csv"
                    candidate_with_uncertainty_df = pd.read_csv(path)
                    candidate_with_uncertainty_df["run_name"] = [file.split("/")[-1] for file in candidate_with_uncertainty_df["cfg:paths:dump_dir"]]
                    pure_cell_type_solution_dict[cell_type].append(candidate_with_uncertainty_df)
                    
                    # uncertainty_path = config_exp_folders / config_exp_folder / "uncertainty_annotation" / "candidates_with_uncertainties.csv"
                    # # Be sure uncertainty annotation exists 
                    # if uncertainty_path.exists():
                    #     candidate_with_uncertainty_df = pd.read_csv(uncertainty_path)
                    #     candidate_with_uncertainty_df["run_name"] = [file.split("/")[-1] for file in candidate_with_uncertainty_df["cfg:paths:dump_dir"]]
                    #     pure_cell_type_solution_dict[cell_type].append(candidate_with_uncertainty_df)

100%|██████████| 6/6 [04:52<00:00, 48.83s/it]


In [8]:
# Concatenate data frame 
pure_cell_type_solution_dict_concat = {
    key: pd.concat(val, axis=0) for key, val in pure_cell_type_solution_dict.items()
}

In [11]:
for col in pure_cell_type_solution_dict_concat["HSCs"].columns:
    print(col)

sample_id
gm-csf_[ng_ml]
tpo_[ng_ml]
sr1_[nm]
um171_[nm]
um729_[µm]
scf_[ng_ml]
butyzamide_[nm]
retinoic_acid_[µm]
ldl_[ng_ml]
il3_[ng_ml]
o2_[%]
days_of_culture
gm-csf_[ng_ml]:rescaled
tpo_[ng_ml]:rescaled
sr1_[nm]:rescaled
um171_[nm]:rescaled
um729_[µm]:rescaled
scf_[ng_ml]:rescaled
butyzamide_[nm]:rescaled
retinoic_acid_[µm]:rescaled
ldl_[ng_ml]:rescaled
il3_[ng_ml]:rescaled
o2_[%]:rescaled
days_of_culture:rescaled
loss
CD14Mono_prop
CD16Mono_prop
CyclingPro1_prop
CyclingPro2_prop
Early_GMP_prop
EosBasMast1_prop
EosBasMast2_prop
EosBasMast3_prop
EryPro_prop
GMP-Neutro_prop
GMP_cycling_prop
HSCs_prop
LMPP-CLP_prop
MLP_prop
MPP_prop
MPP_MgkEry_prop
MgKEryPro1_prop
MgKEryPro2_prop
ProB_prop
earlyMPP_prop
late_MgkPro_prop
preProB_prop
pre_cDC1_prop
pre_cDC2_prop
cfg:annotation:protocol_columns
cfg:annotation:protocol_obs_key_added
cfg:annotation:protocol_sep
cfg:annotation:one_hot_uns_key_added
cfg:annotation:protocol_obsm_key
cfg:annotation:scatter_columns
cfg:annotation:base_sample_re

### Initialize targets

In [15]:
# Cell type fractions
celltype_fraction = {cell_type: 0.15 for cell_type in cell_types}

columns_of_interest = [
    'gm-csf_[ng_ml]',
    'tpo_[ng_ml]',
    'sr1_[nm]',
    'um171_[nm]',
    'um729_[µm]',
    'scf_[ng_ml]',
    'butyzamide_[nm]',
    'retinoic_acid_[µm]',
    'ldl_[ng_ml]',
    'il3_[ng_ml]',
    'o2_[%]',
    'days_of_culture',
    'run_name',
    "cfg:constraints:c_scheduler_kwargs:t_warmup",
    "cfg:constraints:c_scheduler_kwargs:vmin",
    "cfg:constraints:c_scheduler_kwargs:vmax",
    # "loss", 
    # "target_ct_loss_std",
    # "target_ct_loss_mean",
    # "ct_prop_total_variance",
    # "exploitation_score",
    # "exploration_score",
    # "weighted_uncertainty_score",
    # "metric_response_surface", 
    # "indices",
    # "losses",
    # "acq_values"
]

In [27]:
for cell_type in cell_types:
    print(f"{cell_type} proportion", pure_cell_type_solution_dict_concat[cell_type][f"{cell_type}_prop"].max())

# pure_cell_type_solution_dict_concat["CD14Mono"]["CD14Mono_prop"].max()

CD14Mono proportion 0.013714729
CD16Mono proportion 0.023602193
CyclingPro1 proportion 0.98844457
CyclingPro2 proportion 0.0116754025
EosBasMast1 proportion 0.6532045
EosBasMast2 proportion 0.44782972
EosBasMast3 proportion 0.011882929
EryPro proportion 0.28401634
HSCs proportion 0.014655135
ProB proportion 0.0031341102
late_MgkPro proportion 0.8840441
GMP-Neutro proportion 0.37897196
preProB proportion 0.019361315
pre_cDC1 proportion 0.009709132


### Filter values

In [16]:
# Collect filtered and annotated data frames 
filtered_df, annotated_df = manual_filtering(
    df_dict=pure_cell_type_solution_dict_concat,
    bounds=bounds,
    columns_to_keep=columns_of_interest, 
    celltype_fraction=celltype_fraction,
    margin_frac_bounds=0.5, 
    protocol_cols=protocol_cols
)

In [17]:
# Filtered and unfiltered solutions 
for ct in filtered_df:
    print(f"Filtered {ct} number of solutions:", filtered_df[ct].shape[0])
    print(f"Unfiltered {ct} number of solutions:", pure_cell_type_solution_dict_concat[ct].shape[0], "\n")

Filtered CD14Mono number of solutions: 0
Unfiltered CD14Mono number of solutions: 37200 

Filtered CD16Mono number of solutions: 0
Unfiltered CD16Mono number of solutions: 37100 

Filtered CyclingPro1 number of solutions: 6366
Unfiltered CyclingPro1 number of solutions: 27300 

Filtered CyclingPro2 number of solutions: 0
Unfiltered CyclingPro2 number of solutions: 27200 

Filtered EosBasMast1 number of solutions: 7546
Unfiltered EosBasMast1 number of solutions: 27400 

Filtered EosBasMast2 number of solutions: 1685
Unfiltered EosBasMast2 number of solutions: 26900 

Filtered EosBasMast3 number of solutions: 0
Unfiltered EosBasMast3 number of solutions: 27200 

Filtered EryPro number of solutions: 6989
Unfiltered EryPro number of solutions: 32100 

Filtered HSCs number of solutions: 0
Unfiltered HSCs number of solutions: 32400 

Filtered ProB number of solutions: 0
Unfiltered ProB number of solutions: 37300 

Filtered late_MgkPro number of solutions: 4308
Unfiltered late_MgkPro number o